In [1]:
import sys
import os

# Thêm đường dẫn tới thư mục gốc của ai-service để có thể import các module
sys.path.append(os.path.abspath('..'))

import pandas as pd
import joblib

# 1. Load mô hình đã huấn luyện
model_path = "../models/hybrid_markov_model.pkl"
try:
    artifacts = joblib.load(model_path)
    markov_matrix = artifacts["markov_matrix"]
    brand_profile = artifacts["brand_profile"]
    segment_profile = artifacts["segment_profile"]
    global_popular = artifacts.get("global_popular", [])
    print("✅ Load mô hình thành công!")
except Exception as e:
    print(f"⚠️ Lỗi khi load mô hình: {e}")

✅ Load mô hình thành công!


In [2]:
# 2. Định nghĩa hàm dự đoán để thử nghiệm
def predict_next_products(user_data):
    cat_state = int(user_data['last_purchased_category_id'])
    brand_state = str(user_data['favourite_brand'])
    segment_state = str(user_data['price_segment'])
    
    if not markov_matrix:
        return []

    # Xử lý trường hợp danh mục không tồn tại trong ma trận -> Chọn fallback category 4
    if cat_state not in markov_matrix:
        cat_state = 4
    
    scores = {prod: prob for prod, prob in markov_matrix[cat_state].items()}

    # Cộng điểm theo Brand
    if brand_state in brand_profile:
        for prod in brand_profile[brand_state]:
            if prod in scores: scores[prod] += 0.25

    # Cộng điểm theo Phân khúc giá
    if segment_state in segment_profile:
        for prod in segment_profile[segment_state]:
            if prod in scores: scores[prod] += 0.20

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    recommendations = []
    for prod, prob in sorted_scores[:4]:
        recommendations.append({"productId": prod, "confidenceScore": round(min(prob, 1.0), 2)})
        
    # Cold start: Thêm sản phẩm trending nếu thiếu
    if len(recommendations) < 4:
         for pop in global_popular:
             if len(recommendations) >= 4:
                 break
             if not any(r["productId"] == pop for r in recommendations):
                 recommendations.append({"productId": int(pop), "confidenceScore": 0.01})
                 
    return recommendations

In [3]:
# 3. Chạy thử nghiệm với một bộ dữ liệu giả lập (Mock Data)
test_users = [
    {
        "user_id": 99,
        "name": "Người mua Laptop cao cấp",
        "last_purchased_category_id": 2, # Vừa mua danh mục 2 (có thể là đồ công nghệ)
        "favourite_brand": "Apple",
        "price_segment": "HIGH"
    },
    {
        "user_id": 100,
        "name": "Người mua đồ tầm trung, thích Asus",
        "last_purchased_category_id": 1,
        "favourite_brand": "Asus",
        "price_segment": "MEDIUM"
    },
    {
        "user_id": 101,
        "name": "Người dùng tinh gọn, thích đồ giá rẻ",
        "last_purchased_category_id": 6,
        "favourite_brand": "Xiaomi",
        "price_segment": "LOW"
    }
]

print("============= KẾT QUẢ DỰ ĐOÁN =============\n")
for user in test_users:
    recs = predict_next_products(user)
    print(f"👤 {user['name']} (Brand: {user['favourite_brand']}, Giá: {user['price_segment']}, Đã mua DM: {user['last_purchased_category_id']})")
    print(f"   => Gợi ý cho người dùng:")
    for rec in recs:
        print(f"       • Sản phẩm ID: {rec['productId']} - Độ phù hợp: {rec['confidenceScore']*100:.0f}%")
    print("-" * 50)

============= KẾT QUẢ DỰ ĐOÁN =============

👤 Người mua Laptop cao cấp (Brand: Apple, Giá: HIGH, Đã mua DM: 2)
   => Gợi ý cho người dùng:
       • Sản phẩm ID: 8 - Độ phù hợp: 54%
       • Sản phẩm ID: 21 - Độ phù hợp: 54%
       • Sản phẩm ID: 22 - Độ phù hợp: 54%
       • Sản phẩm ID: 3 - Độ phù hợp: 45%
--------------------------------------------------
👤 Người mua đồ tầm trung, thích Asus (Brand: Asus, Giá: MEDIUM, Đã mua DM: 1)
   => Gợi ý cho người dùng:
       • Sản phẩm ID: 18 - Độ phù hợp: 70%
       • Sản phẩm ID: 14 - Độ phù hợp: 57%
       • Sản phẩm ID: 16 - Độ phù hợp: 45%
       • Sản phẩm ID: 1 - Độ phù hợp: 33%
--------------------------------------------------
👤 Người dùng tinh gọn, thích đồ giá rẻ (Brand: Xiaomi, Giá: LOW, Đã mua DM: 6)
   => Gợi ý cho người dùng:
       • Sản phẩm ID: 14 - Độ phù hợp: 45%
       • Sản phẩm ID: 28 - Độ phù hợp: 33%
       • Sản phẩm ID: 5 - Độ phù hợp: 20%
       • Sản phẩm ID: 7 - Độ phù hợp: 20%
----------------------------------